# Second Model - Random Forest Regressor

**Why this model?**

Our research question is not which model predicts best, but whether the gap between random, blocked and chronological validation depends on the kind of model. Ridge and gradient boosting sit at two ends of that question: one is a rigid linear model, the other a flexible tree ensemble that corrects its own errors step by step. A random forest sits between them and works by a different mechanism.

A forest grows many deep trees independently, each on a bootstrap sample of rows and with a random subset of features at every split, and averages their predictions. Averaging reduces the variance of the individual trees, whereas boosting reduces bias. Like boosting, it can pick up the interaction between hour and day type on its own and needs no manual feature engineering. Unlike boosting, it has few sensitive hyperparameters and does not overfit as more trees are added, which makes it a robust second tree-based model.

The main reason to include it is a hypothesis about memorisation. With small leaves a forest can almost reproduce its training data, so under random k-fold it may rely on neighbouring hours that sit in both train and test. If the gap between schemes is a general property of flexible models, the forest should show it as well; if it is specific to boosting, the forest should behave differently. We also record which `min_samples_leaf` and `max_features` each scheme selects, to see whether the schemes favour different model complexity, without fixing an expectation about the direction in advance.

The forest shares one limitation with boosting: trees cannot predict outside the range of the training targets, so the higher demand of 2012 may be underestimated. This notebook does not measure that bias. Missing values of `hum` are imputed with the median inside the pipeline, as in the Ridge notebook, so nothing from the validation part leaks into the imputer.

In [1]:
import ast
import time

import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

from src.data import load_data
from src.validation import RESULTS_DIR, SCHEMES, SEED, evaluate, get_cv

In [2]:
X, y = load_data()

assert X.index.equals(y.index)
assert X.index.is_monotonic_increasing
assert X.index.is_unique
assert "yr" not in X.columns
assert X["hum"].isna().sum() == 22

print("X shape:", X.shape, "| y shape:", y.shape)
print("Features:", X.columns.tolist())
print("Validation schemes:", SCHEMES)

X shape: (17379, 9) | y shape: (17379,)
Features: ['mnth', 'hr', 'holiday', 'weekday', 'workingday', 'weathersit', 'temp', 'hum', 'windspeed']
Validation schemes: ('random', 'blocked', 'chrono')


## Preprocessing

Unlike linear models, tree-based ensembles are invariant to monotonic feature scaling and do not require one-hot encoding of categorical indicators. The model directly ingests the processed tabular features (`mnth`, `hr`, `holiday`, `weekday`, `workingday`, `weathersit`, `temp`, `hum`, `windspeed`).

The 22 missing values in `hum` are imputed using `SimpleImputer(strategy="median")` inside a scikit-learn `Pipeline`, so the median is computed only on the training part of each split (outer and inner) and nothing from the validation part leaks into the imputer. Recent scikit-learn versions can also route missing values natively in random forests, as `HistGradientBoostingRegressor` does. We use explicit imputation to stay consistent with the Ridge setup, which means the three models do not handle missing values in exactly the same way.

In [3]:
N_ESTIMATORS = 200

rf_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestRegressor(n_estimators=N_ESTIMATORS, random_state=SEED, n_jobs=1)),
])

## Random Forest and Hyperparameters

Hyperparameter selection focuses on governing tree complexity and ensemble decorrelation:

- `n_estimators`: Fixed at 200 trees. Ensembles governed by bagging do not overfit as the tree count increases; adding trees generally reduces estimation variance, with diminishing returns, at the expense of compute time.
- `min_samples_leaf`: Serves as the primary complexity and smoothing control (`[1, 5, 20]`). A value of 1 permits individual hour memorization, while higher thresholds prevent trees from isolating noisy temporal spikes.
- `max_features`: Controls feature subspace sampling (`[0.5, 1.0]`). Evaluating a random subset (50%) at each split decorrelates individual trees, which tends to make the ensemble more robust.
- `max_depth`: Left unconstrained (`None`), as regularization is governed explicitly via `min_samples_leaf`.
- `oob_score`: Deliberately set to `False`. Out-of-bag scoring relies on random row-level bootstrapping, which violates the strict non-overlapping temporal partitioning of chronological splits.

In [4]:
PARAM_GRID = {
    "min_samples_leaf": [1, 5, 20],
    "max_features": [0.5, 1.0],
}
INNER_SPLITS = 3


def make_rf_search(scheme):
    """RF pipeline tuned with an inner CV of the same type as the outer scheme."""
    return GridSearchCV(
        rf_pipeline,
        {f"model__{k}": v for k, v in PARAM_GRID.items()},
        cv=get_cv(scheme, n_splits=INNER_SPLITS),
        scoring="neg_root_mean_squared_error",
        n_jobs=-1,
    )

## Computational Cost

A single hyperparameter search over 6 candidate configurations ($3 \times 2$) with 3 inner cross-validation folds takes about 26.6 seconds on the complete dataset. The full nested run (5 outer folds for each of the 3 schemes) took about 4 minutes in total, roughly 1.3 to 1.4 minutes per scheme (see the output of the run below), so the grid and the 200 trees are computationally practical.

In [5]:
start = time.perf_counter()
benchmark = make_rf_search("random").fit(X, y)
print(f"One search on all data: {time.perf_counter() - start:.1f} s")
print("Best parameters:", benchmark.best_params_)

One search on all data: 26.6 s
Best parameters: {'model__max_features': 0.5, 'model__min_samples_leaf': 1}


## Validation Strategy and Hyperparameter Tuning

We implement a strictly nested cross-validation protocol using the three core evaluation schemes defined in `src/validation.py`:

- **random**: Standard 5-fold cross-validation with random row shuffling. Adjacent temporal points appear across both training and validation splits.
- **blocked**: Contiguous non-overlapping temporal blocks evaluated sequentially. No shuffling is performed, though training sets include both preceding and subsequent observations.
- **chrono**: Forward-chaining expanding window over 2012 test blocks. The model is trained strictly on historical data prior to the evaluation window, simulating production forecasting.

Hyperparameters are optimized inside each outer training fold using `GridSearchCV` over 3 inner splits matched to the respective outer scheme type. Once the optimal parameter configuration is identified via negative root mean squared error, the pipeline is refitted on the entire outer training split and evaluated against the unseen outer test fold.

One caveat concerns the early chronological folds. Their training part covers only about one year, so the inner chronological splits validate on months that the inner training part has not seen. This is a limitation of the data, not of the code, and it may make the hyperparameter choice in those folds less reliable.

In [6]:
variants = ["rf"]

results = {}
for name in variants:
    for scheme in SCHEMES:
        start = time.perf_counter()
        model = make_rf_search(scheme)
        folds, preds = evaluate(model, X, y, scheme, name)
        results[(name, scheme)] = (folds, preds)
        print(f"{name:8s} {scheme:8s} {(time.perf_counter() - start) / 60:5.2f} min   "
              f"mean RMSE {folds['rmse'].mean():.1f}   mean MAE {folds['mae'].mean():.1f}")

rf       random    1.27 min   mean RMSE 65.8   mean MAE 42.9
rf       blocked   1.28 min   mean RMSE 114.1   mean MAE 80.6
rf       chrono    1.44 min   mean RMSE 117.1   mean MAE 83.1


## Experimental Results

The table below summarizes the performance metrics averaged across outer folds for both baseline estimators and the Random Forest regressor (`rf`). The metrics include Root Mean Squared Error (RMSE), Mean Absolute Error (MAE), Coefficient of Determination ($R^2$), and Root Mean Squared Logarithmic Error (RMSLE). 

Because the chronological scheme evaluates strictly on 2012 test blocks whereas random and blocked cross-validations span the entirety of 2011–2012, fold-averaged metrics must only be used to rank competing models within a given scheme, rather than to measure the cross-validation gap directly.

In [7]:
def summarise(name, scheme):
    folds = pd.read_csv(RESULTS_DIR / f"{name}_{scheme}_folds.csv")
    return {
        "model": name,
        "scheme": scheme,
        "rmse_mean": folds["rmse"].mean(),
        "rmse_std": folds["rmse"].std(),
        "mae_mean": folds["mae"].mean(),
        "r2_mean": folds["r2"].mean(),
        "rmsle_mean": folds["rmsle"].mean(),
    }


names = ["baseline_const", "baseline_hour_day", *variants]
pd.DataFrame([summarise(n, s) for n in names for s in SCHEMES]).round(3)

,model,scheme,rmse_mean,rmse_std,mae_mean,r2_mean,rmsle_mean
0,baseline_const,random,181.372,3.540,142.415,-0.001,1.570
1,baseline_const,blocked,182.116,47.228,147.169,-0.280,1.575
2,baseline_const,chrono,215.611,52.739,165.455,-0.159,1.443
3,baseline_hour_day,random,109.942,1.384,72.959,0.632,0.648
4,baseline_hour_day,blocked,119.111,38.852,83.458,0.304,0.658
5,baseline_hour_day,chrono,133.292,44.825,95.482,0.562,0.580
6,rf,random,65.812,2.000,42.946,0.868,0.423
7,rf,blocked,114.130,14.554,80.589,0.453,0.563
8,rf,chrono,117.107,11.547,83.148,0.630,0.506


### Reading the table

Within every scheme the forest has a lower RMSE than both baselines, but the margin depends strongly on the scheme. Relative to the hour $\times$ day type baseline it is about 40% lower under random validation (65.8 against 109.9), only about 4% lower under blocked validation (114.1 against 119.1) and about 12% lower under chronological validation (117.1 against 133.3). These values are fold averages, so they should be compared within a scheme rather than across schemes.

## Selected Hyperparameters

The table below lists the configuration chosen by the inner search in each outer fold. The choices are descriptive: we did not examine how close competing configurations were in inner CV score, so we do not interpret them causally.

- **Random validation**: `min_samples_leaf=1` and `max_features=0.5` in all five folds.
- **Blocked validation**: mixed choices. `min_samples_leaf=5` in three of five folds and `1` in the other two; `max_features=1.0` in three folds and `0.5` in two.
- **Chronological validation**: `min_samples_leaf=1` and `max_features=1.0` in all five folds.

Two observations follow. First, `min_samples_leaf` does not separate random from chronological validation (both choose 1, the smallest possible value), so the initial idea that random validation would favour smaller leaves than chronological validation is not supported here. Second, the schemes differ in `max_features` (0.5 under random, 1.0 under chronological), but this experiment cannot tell us why. Note also that `max_features=0.5` is the lower edge of the grid, so a smaller value might have been preferred had it been available.

In [8]:
rows = []
for (name, scheme), (folds, _) in results.items():
    params = pd.DataFrame([ast.literal_eval(s) for s in folds["best_params"]])
    params.columns = [c.split("__")[-1] for c in params.columns]
    rows.append(params.assign(model=name, scheme=scheme, fold=folds["fold"].to_numpy()))

pd.concat(rows).set_index(["model", "scheme", "fold"])

max_features  min_samples_leaf
model scheme  fold                                
rf    random  1              0.5                 1
              2              0.5                 1
              3              0.5                 1
              4              0.5                 1
              5              0.5                 1
      blocked 1              1.0                 5
              2              1.0                 5
              3              1.0                 1
              4              0.5                 1
              5              0.5                 5
      chrono  1              1.0                 1
              2              1.0                 1
              3              1.0                 1
              4              1.0                 1
              5              1.0                 1

## Fold-Level Performance (chronological scheme)

Individual folds show clear differences between periods. The test windows cover different seasons, so the folds are not interchangeable.

- **Folds 1 and 5 (January to mid-March and late October to December 2012)** have the lowest absolute errors ($\text{RMSE} = 105.1$ and $106.0$, $\text{MAE} = 72.6$ and $72.3$). Fold 1 has the smallest training set (2011 only), yet its absolute error is among the lowest, which is consistent with low winter demand. Relative to the signal it is the hardest fold: $R^2$ is lowest ($0.408$) and RMSLE highest ($0.681$).
- **Folds 2 to 4 (mid-March to mid-October 2012)** have higher absolute errors ($\text{RMSE}$ between $119.7$ and $132.2$), peaking in fold 4 ($\text{RMSE} = 132.2$, $\text{MAE} = 94.4$). This may reflect higher demand in spring to autumn 2012, but we did not check whether the test targets exceed the range seen in training.
- **Fold 5** ($R^2 = 0.683$) has the largest training set (15,664 hours), which may help it, but its test period is also a low-demand season, so the two effects cannot be separated here.

In [9]:
for name in variants:
    print(name)
    display(results[(name, "chrono")][0].drop(columns="best_params")
            .round({"rmse": 1, "mae": 1, "r2": 3, "rmsle": 3}))

rf


,fold,n_train,n_test,test_start,test_end,rmse,mae,r2,rmsle
0,1,8657,1746,2012-01-01 12:00:00,2012-03-14 13:00:00,105.1,72.6,0.408,0.681
1,2,10403,1753,2012-03-14 14:00:00,2012-05-26 16:00:00,122.6,87.9,0.664,0.493
2,3,12156,1754,2012-05-26 17:00:00,2012-08-07 18:00:00,119.7,88.5,0.701,0.408
3,4,13910,1754,2012-08-07 19:00:00,2012-10-19 20:00:00,132.2,94.4,0.695,0.417
4,5,15664,1715,2012-10-19 21:00:00,2012-12-31 23:00:00,106.0,72.3,0.683,0.530


## Conclusions

- **Hyperparameter selection**: The inner search chose `min_samples_leaf=1` under random and chronological validation and a mixed set of values under blocked validation (5 in three folds, 1 in two). The expectation that random validation would favour smaller leaves than chronological validation was therefore not confirmed. The schemes differ in `max_features` (0.5 under random, 1.0 under chronological), but we did not investigate why, and `max_features=0.5` sits on the lower edge of the grid.
- **Validation scheme gap**: Fold-averaged $\text{RMSE}$ rises from $65.8$ (random) to $114.1$ (blocked) and $117.1$ (chrono). These averages are computed on different rows (chronological folds test only 2012, while random and blocked folds cover 2011 to 2012) and on training sets of different size, so they do not measure the gap exactly. They are still informative next to the baseline. The hour $\times$ day type baseline, which cannot exploit neighbouring hours, worsens by only about 8% between random and blocked validation ($109.9$ to $119.1$), while the forest worsens by about 73%. This suggests that most of the forest's optimism under random validation comes from neighbouring hours shared between train and test. Whether the same holds for the other models, compared on identical rows, is left to `06_comparison_and_errors.ipynb`.
- **Blocked against chronological validation**: The step from blocked to chrono is small in RMSE ($114.1$ to $117.1$) and has the opposite sign in RMSLE ($0.563$ to $0.506$), so we do not claim that chronological validation is harder than blocked validation for this model.
- **Comparison with baselines**: Within every scheme the forest beats both baselines in RMSE, by about 40% (random), 4% (blocked) and 12% (chrono) relative to the hour $\times$ day type baseline.
- **Extrapolation**: Trees cannot predict outside the range of the training targets, and demand in 2012 was higher than in 2011, so under-prediction of high-demand hours is plausible, especially in the early chronological folds. This notebook does not measure the residual bias or the errors at peak hours; these checks belong to the error analysis in `06_comparison_and_errors.ipynb`.
- **Limitations**: Chronological folds train on 8,657 to 15,664 hours, while random and blocked folds train on about 13,900. The inner chronological splits in the early folds validate on months not seen in the inner training part. The forest is fitted with a single random seed (`random_state=SEED`), so we have no estimate of seed-to-seed variation. The model uses only static weather and calendar variables, without `yr` or lagged demand, on a two-year observation period.